<a href="https://colab.research.google.com/github/marviiiin/Reproducibility-Learning_To_Edit-/blob/main/ALL_IN_ONE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LTE-LoRA — Train + Evaluate (single notebook)

Reproducibility study for **Jiang et al. 2024, "Learning to Edit" (ACL 2024)**.

**One notebook, one click.** Open in Colab, set runtime to A100, then `Runtime → Run all`. Walk away. ~10 hours.

**What it does:**
1. Sets up environment.
2. Clones LTE repo, downloads SFT data from HuggingFace, applies all 7 patches.
3. Trains LTE-LoRA on LLaMA-2-Chat-7B (1 epoch, seq 1024, bf16, A100 ~6 hr).
4. Saves adapter to Drive.
5. Sanity check.
6. Evaluates on ZsRE (4 aspects) and WikiData_counterfact (5 aspects), 200 examples each.
7. Aggregates and saves results.

**Compute units:** ~50 (Colab Pro A100 at 5.37/hr × ~9.5 hr).

**Note:** authors didn't release a pre-trained LTE-LoRA checkpoint, only the SFT data. Training is unavoidable.

## Cell 1 — GPU sanity check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100.'
name = torch.cuda.get_device_name(0)
print(f'GPU: {name}')
if 'A100' not in name:
    print('WARNING: not on A100. Will be slower and more expensive.')

## Cell 2 — Install dependencies

Includes everything for training (FastChat) and evaluation (EasyEdit). flash_attn skipped (15 min compile).

In [ ]:
!pip install -q peft bitsandbytes scipy deepspeed sentence_transformers higher
!pip install -q fschat
!pip install -q iopath omegaconf einops timm fairscale hydra-core nltk sentencepiece protobuf openai matplotlib
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('OK')

## Cell 3 — Clone repo, download data, apply ALL 7 patches

**FastChat patches** (transformers >= 4.46 compatibility):
  1. `from transformers import deepspeed` moved to `transformers.integrations`
  2. `flash_attn_monkey_patch` import optional
  3. `CPUAdamBuilder().load()` optional
  4. `Trainer(tokenizer=)` -> `Trainer(processing_class=)` (surgical, only Trainer call)
  5. `deepspeed.is_deepspeed_zero3_enabled()` bypass (function moved)

**EasyEdit patches:**
  6. Stub out `blip2_models/__init__.py` (vision-language API drift)
  7. Wrap `KnowEditDataset` in `list()` for `random.sample`

In [ ]:
import os, re
os.chdir('/content')
!rm -rf LTE
!git clone -q https://github.com/YJiangcm/LTE.git
os.chdir('/content/LTE')

from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='YuxinJiang/LTE_train_data', filename='LTE_train_data.json',
                repo_type='dataset', local_dir='data/')
print(f'SFT data: {os.path.getsize("data/LTE_train_data.json") / 1e6:.0f} MB')

trainer = 'FastChat/fastchat/train/train_lora.py'
with open(trainer) as f:
    code = f.read()
code = code.replace(
    'from transformers import Trainer, BitsAndBytesConfig, deepspeed',
    'from transformers import Trainer, BitsAndBytesConfig\ntry:\n    from transformers import deepspeed\nexcept ImportError:\n    from transformers.integrations import deepspeed'
)
code = code.replace(
    'from fastchat.train.llama_flash_attn_monkey_patch import (\n    replace_llama_attn_with_flash_attn,\n)',
    'try:\n    from fastchat.train.llama_flash_attn_monkey_patch import (\n        replace_llama_attn_with_flash_attn,\n    )\nexcept ImportError:\n    def replace_llama_attn_with_flash_attn(): pass'
)
code = code.replace(
    'import deepspeed\ndeepspeed.ops.op_builder.CPUAdamBuilder().load()',
    'try:\n    import deepspeed\n    deepspeed.ops.op_builder.CPUAdamBuilder().load()\nexcept Exception:\n    pass'
)
code = re.sub(r'Trainer\(\s*model=model,\s*tokenizer=tokenizer',
              'Trainer(model=model, processing_class=tokenizer', code)
code = code.replace('deepspeed.is_deepspeed_zero3_enabled()', 'False')
with open(trainer, 'w') as f:
    f.write(code)

with open('/content/LTE/EasyEdit/easyeditor/trainer/blip2_models/__init__.py', 'w') as f:
    f.write('# Stubbed for text-only editing\n')

rk = '/content/LTE/EasyEdit/run_knowedit.py'
with open(rk) as f:
    code = f.read()
code = code.replace('datas = random.sample(datas, args.ds_size)',
                    'datas = random.sample(list(datas), args.ds_size)')
with open(rk, 'w') as f:
    f.write(code)

print('All 7 patches applied.')

## Cell 4 — Pre-download model weights

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
MODEL = 'NousResearch/Llama-2-7b-chat-hf'
tok = AutoTokenizer.from_pretrained(MODEL)
m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
print(f'{MODEL}: {m.num_parameters()/1e9:.1f}B params')
del m, tok
torch.cuda.empty_cache()

## Cell 5 — TRAIN (~6 hours on A100)

1 epoch, seq 1024, eff batch 128 (per_device 2 × accum 64), lr 3e-4 cosine, bf16, gradient checkpointing.

In [ ]:
import os, torch
os.chdir('/content/LTE')
os.environ['WANDB_DISABLED'] = 'true'
torch.cuda.empty_cache()

!python FastChat/fastchat/train/train_lora.py \
    --model_name_or_path NousResearch/Llama-2-7b-chat-hf \
    --lora_r 8 --lora_alpha 16 --lora_dropout 0.05 \
    --data_path data/LTE_train_data.json \
    --bf16 True \
    --output_dir output_lte_lora_llama-2_7b_chat \
    --num_train_epochs 1 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 64 \
    --eval_strategy no \
    --save_strategy steps --save_steps 100 --save_total_limit 3 \
    --learning_rate 3e-4 --weight_decay 0.0 --warmup_ratio 0.03 \
    --lr_scheduler_type cosine --logging_steps 10 \
    --tf32 True --model_max_length 1024 \
    --gradient_checkpointing True \
    --q_lora False --report_to none

## Cell 6 — Recover adapter from latest checkpoint and save to Drive

If the post-training save errored, recover from the highest checkpoint dir.

In [ ]:
import os, glob, shutil
out = '/content/LTE/output_lte_lora_llama-2_7b_chat'
if not os.path.exists(os.path.join(out, 'adapter_config.json')):
    ckpts = sorted(glob.glob(f'{out}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
    assert ckpts, 'No checkpoints found.'
    latest = ckpts[-1]
    print(f'Recovering from {latest}')
    for f in ['adapter_config.json', 'adapter_model.safetensors',
              'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json']:
        src = os.path.join(latest, f)
        if os.path.exists(src):
            shutil.copy(src, out)
assert os.path.exists(os.path.join(out, 'adapter_config.json'))
for f in sorted(os.listdir(out)):
    full = os.path.join(out, f)
    if os.path.isfile(full):
        print(f'  {f}  ({os.path.getsize(full)/1e6:.1f} MB)')
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/LTE_checkpoints
!cp -r /content/LTE/output_lte_lora_llama-2_7b_chat /content/drive/MyDrive/LTE_checkpoints/
print('Adapter saved to Drive.')

## Cell 7 — One-example sanity check ([INST] format)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
BASE = 'NousResearch/Llama-2-7b-chat-hf'
ADAPTER = '/content/LTE/output_lte_lora_llama-2_7b_chat'
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()
print('LoRA active:', hasattr(model, 'peft_config'))
print('LoRA target modules:', model.peft_config['default'].target_modules)
USER_MSG = ('Please acknowledge the updated information provided below and respond to the subsequent query.\n\n'
            '[Updated Information]:\n1. The capital of France is Berlin.\n\n[Query]:\nWhat is the capital of France?')
PROMPT = f'[INST] {USER_MSG} [/INST]'
inp = tok(PROMPT, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inp, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
print('Model says:', repr(tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)))
del model, tok; torch.cuda.empty_cache()

## Cell 8 — Set up evaluation (retrieval encoder + hparams)

In [ ]:
import os, yaml
os.chdir('/content/LTE')
if not os.path.exists('hugging_cache/all-MiniLM-L6-v2'):
    !git clone -q https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 hugging_cache/all-MiniLM-L6-v2
hp = '/content/LTE/EasyEdit/hparams/IKE/llama-7b.yaml'
with open(hp) as f:
    cfg = yaml.safe_load(f)
cfg['model_name'] = 'NousResearch/Llama-2-7b-chat-hf'
cfg['lora_name'] = '/content/LTE/output_lte_lora_llama-2_7b_chat'
cfg['sentence_model_name'] = '/content/LTE/hugging_cache/all-MiniLM-L6-v2'
with open(hp, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Hparams updated:', cfg)

## Cell 9 — Evaluate LTE-LoRA on ZsRE (~1.7 hr, 200 ex/aspect)

In [ ]:
import os
os.chdir('/content/LTE/EasyEdit')
os.makedirs('output', exist_ok=True)
for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs']:
    print(f'\n{"="*20}  ZsRE / {aspect}  {"="*20}')
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/ZsRE/ZsRE-test-all.json \
        --eval_aspect=$aspect --fluency --ds_size 200

## Cell 10 — Evaluate LTE-LoRA on WikiData_counterfact (~2.1 hr, 200 ex/aspect)

In [ ]:
import os
os.chdir('/content/LTE/EasyEdit')
for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs', 'locality_f']:
    print(f'\n{"="*20}  wiki_counterfact / {aspect}  {"="*20}')
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/wiki_counterfact/test_cf.json \
        --eval_aspect=$aspect --fluency --ds_size 200

## Cell 11 — Aggregate metrics into spreadsheet-ready JSON

In [ ]:
import json, os
OUT_DIR = '/content/LTE/EasyEdit/output'
ASPECT_LABELS = {
    'portability_r':  ('portability', 'Reasoning_acc',             'Portability \u2014 Reasoning'),
    'portability_s':  ('portability', 'Subject_Aliasing_acc',      'Portability \u2014 Subject Aliasing'),
    'portability_l':  ('portability', 'Logical_Generalization_acc','Portability \u2014 Logical Generalization'),
    'locality_rs':    ('locality',    'Relation_Specificity_acc',  'Locality \u2014 Relation Specificity'),
    'locality_f':     ('locality',    'Forgetfulness_acc',         'Locality \u2014 Forgetfulness'),
}
def safe_avg(metrics, path):
    vals = []
    for m in metrics:
        cur = m
        for k in path:
            cur = cur.get(k, None) if isinstance(cur, dict) else None
            if cur is None: break
        if cur is None: continue
        if isinstance(cur, list):
            cur = cur[0] if cur else None
        if cur is not None:
            vals.append(cur)
    return round(100 * sum(vals) / len(vals), 2) if vals else None

results = {'method': 'LTE-LoRA', 'backbone': 'LLaMA-2-Chat-7B',
           'training_notes': '1 epoch, model_max_length=1024',
           'eval_notes': '200-example random subsample per aspect',
           'datasets': {}}
for fname in sorted(os.listdir(OUT_DIR)):
    if not fname.endswith('_results.json'): continue
    body = fname[:-len('_results.json')]
    parts = body.split('_')
    aspect = '_'.join(parts[-2:]) if parts[-2] in {'portability','locality'} else parts[-1]
    dataset = '_'.join(parts[1:-2]) if parts[-2] in {'portability','locality'} else '_'.join(parts[1:-1])
    if aspect not in ASPECT_LABELS: continue
    with open(os.path.join(OUT_DIR, fname)) as f:
        metrics = json.load(f)
    cat, key, label = ASPECT_LABELS[aspect]
    val = safe_avg(metrics, ['post', cat, key])
    es  = safe_avg(metrics, ['post', 'rewrite_acc'])
    flu = safe_avg(metrics, ['post', 'fluency', 'ngram_entropy'])
    bucket = results['datasets'].setdefault(dataset, {})
    bucket[label] = val
    if bucket.get('Edit Success') is None: bucket['Edit Success'] = es
    if bucket.get('Fluency') is None:      bucket['Fluency'] = flu
    print(f'  {dataset:<22} {label:<40} {val}')

out_path = '/content/LTE/EasyEdit/output/lte_lora_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print('\n=== FINAL RESULTS ===')
print(json.dumps(results, indent=2))

## Cell 12 — Save everything to Drive

In [ ]:
import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/LTE_checkpoints/results
!cp /content/LTE/EasyEdit/output/*.json /content/drive/MyDrive/LTE_checkpoints/results/
print('Saved.')
!ls -la /content/drive/MyDrive/LTE_checkpoints/results/

---
## Done

Open `lte_lora_results.json` from Drive and copy numbers into `Table1_Replica.xlsx`.